# HeatRx: Brooklyn data and model check
Run in a Colab Enterprise runtime with access to the hackathon project. This notebook reads the existing tables and model; it does not reload data or retrain anything. Offsets describe the campaign, not today's weather.

In [ ]:
from google.cloud import bigquery

project = input('Google Cloud project ID: ').strip()
client = bigquery.Client(project=project, location='US')
def show(sql):
    return client.query_and_wait(sql, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=256_000_000)).to_dataframe(create_bqstorage_client=False)

In [ ]:
show(f'''SELECT table_name, ordinal_position, column_name, data_type, is_nullable
FROM `{project}.a4i_brooklyn.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN ('heat_observations','station_daily_temps','tract_demographics')
ORDER BY table_name, ordinal_position''')

In [ ]:
show(f'''SELECT COUNT(measured_offset_f) AS measured_cells, AVG(measured_offset_f) AS mean_measured_offset_f, MAX(measured_offset_f) AS max_measured_offset_f
FROM `{project}.a4i_grid.cell_profile_full`
WHERE borough = 'Brooklyn' AND is_land AND tract_geo_id = '36047036900' ''')

In [ ]:
show(f'''SELECT cell_i, cell_j, offset_f AS measured_offset_f, predicted_offset_f, nbr_n, top_feature_attributions
FROM ML.EXPLAIN_PREDICT(MODEL `{project}.a4i_grid.heat_offset_v1`,
    (SELECT * FROM `{project}.a4i_grid.cell_model_input` WHERE cell_i = 40681 AND cell_j = -73909),
    STRUCT(5 AS top_k_features))''')